In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# Vision Transformer MLP

class ViTMLP(nn.Module):
    
    def __init__(
        self,
        mlp_num_hiddens: int,
        mlp_num_outputs: int,
        dropout: float = 0.5,
    ) -> None:
        super().__init__()
        
        self.dense1 = nn.LazyLinear(
            out_features=mlp_num_hiddens,
        )
        
        self.gelu = nn.GELU()
        
        self.dropout1 = nn.Dropout(
            p=dropout,
        )
        
        self.dense2 = nn.LazyLinear(
            out_features=mlp_num_outputs,
        )
        
        self.dropout2 = nn.Dropout(
            p=dropout,
        )
        
        
    def forward(
        self,
        X: torch.Tensor, # [B, T, D]
    ) -> torch.Tensor:
        
        X = self.dense1(X)
        X = self.gelu(X)
        X = self.dropout1(X)
        X = self.dense2(X)
    
        # [B, T, D]
        return self.dropout2(X)

In [3]:
# ViT MLP shape

# [B, T, D]
X = torch.ones(
    (
        2,
        100,
        24,
    )
)

mlp = ViTMLP(
    mlp_num_hiddens=48,
    mlp_num_outputs=24,
    dropout=0.5,
)

mlp.eval()

with torch.no_grad():
    mlp_output = mlp(X)
    
    print(
    "MLP input:",
    tuple(X.shape),
)

print(
    "MLP output:",
    tuple(mlp_output.shape),
)


d2l.check_shape(
    mlp_output,
    X.shape,
)

MLP input: (2, 100, 24)
MLP output: (2, 100, 24)


In [4]:
# Vision Transformer Encoder Block

class ViTBlock(nn.Module):
    
    def __init__(
        self,
        num_hiddens: int,
        norm_shape: int,
        mlp_num_hiddens: int,
        num_heads: int,
        dropout: float,
        use_bias: bool = False,
    ) -> None:
        super().__init__()
        
        # Attention 직전에 적용하는 Pre-Normalization
        self.layer_norm1 = nn.LayerNorm(
            normalized_shape=norm_shape,
        )
        
        self.attention = d2l.MultiHeadAttention(
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
            bias=use_bias,
        )
        
        # MLP 직전에 적용하는 Pre-Normalization
        self.layer_norm2 = nn.LayerNorm(
            normalized_shape=norm_shape,
        )
        
        self.mlp = ViTMLP(
            mlp_num_hiddens=mlp_num_hiddens,
            mlp_num_outputs=num_hiddens,
            dropout=dropout,
        )
        
    def forward(
        self,
        X: torch.Tensor,
        valid_lens: torch.Tensor | None = None,
    ) -> torch.Tensor:
        
        # Pre-Normalization
        normalized_X = self.layer_norm1(
            X
        )
        
        # Self-Attention이므로 Q = K = V
        attention_output = self.attention(
            queries=normalized_X,
            keys=normalized_X,
            values=normalized_X,
            valid_lens=valid_lens,
        )
        
        # First Residual Connection
        X = X + attention_output
        
        # Second Pre-Normalization
        normalized_X = self.layer_norm2(
            X
        )
        
        # Second Residual Connection
        return X + self.mlp(
            normalized_X
        )

In [5]:
# ViT Block shape 검증

batch_size = 2
num_steps = 100
num_hiddens = 24

mlp_num_hiddens = 48
num_heads = 8


X = torch.ones(
    (
        batch_size,
        num_steps,
        num_hiddens,
    )
)


encoder_block = ViTBlock(
    num_hiddens=num_hiddens,
    norm_shape=num_hiddens,
    mlp_num_hiddens=mlp_num_hiddens,
    num_heads=num_heads,
    dropout=0.5,
)

encoder_block.eval()


with torch.no_grad():
    block_output = encoder_block(
        X
    )


print(
    "Block input:",
    tuple(X.shape),
)

print(
    "Block output:",
    tuple(block_output.shape),
)


d2l.check_shape(
    block_output,
    X.shape,
)

Block input: (2, 100, 24)
Block output: (2, 100, 24)
